In [ ]:
import os
import torch

os.environ['TORCH'] = torch.__version__
print(torch.__version__)

2.3.1+cu121


In [2]:
import torch
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GCNConv
from torch_geometric.utils import train_test_split_edges

# Tutorial 6  
Graph AutoEncoders GAE &  
Variational Graph Autoencoders VGAE    

[paper](https://arxiv.org/pdf/1611.07308.pdf)  
[code](https://github.com/rusty1s/pytorch_geometric/blob/master/examples/autoencoder.py)

## Graph AutoEncoder GAE

### Load the data

In [ ]:
dataset = Planetoid('data', 'CiteSeer', transform=T.NormalizeFeatures())
dataset.data

Processing...
Done!
/home/kklepikov/_code_/PytorchGeometricTutorial/.venv/lib/python3.12/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


Data(x=[3327, 3703], edge_index=[2, 9104], y=[3327], train_mask=[3327], val_mask=[3327], test_mask=[3327])

In [8]:
data = dataset[0]
data.train_mask = data.val_mask = data.test_mask = None
data

Data(x=[3327, 3703], edge_index=[2, 9104], y=[3327])

In [9]:
data = train_test_split_edges(data)

/home/kklepikov/_code_/PytorchGeometricTutorial/.venv/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


In [10]:
data

Data(x=[3327, 3703], y=[3327], val_pos_edge_index=[2, 227], test_pos_edge_index=[2, 455], train_pos_edge_index=[2, 7740], train_neg_adj_mask=[3327, 3327], val_neg_edge_index=[2, 227], test_neg_edge_index=[2, 455])

### Define the Encoder

In [ ]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GCNEncoder, self).__init__()
        self.conv1 = GCNConv(
            in_channels, 2 * out_channels, cached=True
        )   # cached only for transductive learning
        self.conv2 = GCNConv(
            2 * out_channels, out_channels, cached=True
        )   # cached only for transductive learning

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

### Define the Autoencoder

In [12]:
from torch_geometric.nn import GAE

In [13]:
# parameters
out_channels = 2
num_features = dataset.num_features
epochs = 100

# model
model = GAE(GCNEncoder(num_features, out_channels))

# move to GPU (if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
x = data.x.to(device)
train_pos_edge_index = data.train_pos_edge_index.to(device)

# inizialize the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
def train():
    model.train()
    optimizer.zero_grad()
    z = model.encode(x, train_pos_edge_index)
    loss = model.recon_loss(z, train_pos_edge_index)
    # if args.variational:
    #   loss = loss + (1 / data.num_nodes) * model.kl_loss()
    loss.backward()
    optimizer.step()
    return float(loss)


def test(pos_edge_index, neg_edge_index):
    model.eval()
    with torch.no_grad():
        z = model.encode(x, train_pos_edge_index)
    return model.test(z, pos_edge_index, neg_edge_index)

In [16]:
for epoch in range(1, epochs + 1):
    loss = train()

    auc, ap = test(data.test_pos_edge_index, data.test_neg_edge_index)
    print('Epoch: {:03d}, AUC: {:.4f}, AP: {:.4f}'.format(epoch, auc, ap))

Epoch: 001, AUC: 0.6333, AP: 0.6790
Epoch: 002, AUC: 0.6488, AP: 0.6910
Epoch: 003, AUC: 0.6583, AP: 0.6971
Epoch: 004, AUC: 0.6632, AP: 0.7012
Epoch: 005, AUC: 0.6676, AP: 0.7052
Epoch: 006, AUC: 0.6710, AP: 0.7090
Epoch: 007, AUC: 0.6740, AP: 0.7126
Epoch: 008, AUC: 0.6762, AP: 0.7147
Epoch: 009, AUC: 0.6780, AP: 0.7167
Epoch: 010, AUC: 0.6793, AP: 0.7188
Epoch: 011, AUC: 0.6802, AP: 0.7209
Epoch: 012, AUC: 0.6794, AP: 0.7227
Epoch: 013, AUC: 0.6784, AP: 0.7240
Epoch: 014, AUC: 0.6776, AP: 0.7262
Epoch: 015, AUC: 0.6762, AP: 0.7280
Epoch: 016, AUC: 0.6746, AP: 0.7295
Epoch: 017, AUC: 0.6726, AP: 0.7304
Epoch: 018, AUC: 0.6711, AP: 0.7312
Epoch: 019, AUC: 0.6697, AP: 0.7313
Epoch: 020, AUC: 0.6687, AP: 0.7317
Epoch: 021, AUC: 0.6681, AP: 0.7322
Epoch: 022, AUC: 0.6678, AP: 0.7325
Epoch: 023, AUC: 0.6674, AP: 0.7327
Epoch: 024, AUC: 0.6672, AP: 0.7330
Epoch: 025, AUC: 0.6673, AP: 0.7334
Epoch: 026, AUC: 0.6673, AP: 0.7337
Epoch: 027, AUC: 0.6675, AP: 0.7340
Epoch: 028, AUC: 0.6687, AP:

In [17]:
Z = model.encode(x, train_pos_edge_index)
Z

tensor([[ 0.4194,  0.2287],
        [-1.1365, -0.7346],
        [ 0.8070,  0.4928],
        ...,
        [-0.6755, -0.4429],
        [ 0.7948,  0.4824],
        [ 1.0247,  0.6925]], device='cuda:0', grad_fn=<AddBackward0>)

## Are the results (AUC) and (AP) easy to read and compare?

# Use Tensorboard

In [19]:
from torch.utils.tensorboard import SummaryWriter

In [20]:
# parameters
out_channels = 2
num_features = dataset.num_features
epochs = 100

# model
model = GAE(GCNEncoder(num_features, out_channels))

# move to GPU (if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
x = data.x.to(device)
train_pos_edge_index = data.train_pos_edge_index.to(device)

# inizialize the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

### Import tensorboard

#### Installation: (if needed) "pip install tensorboard"

In [ ]:
writer = SummaryWriter('runs/GAE1_experiment_' + '2d_100_epochs')

In [ ]:
for epoch in range(1, epochs + 1):
    loss = train()
    auc, ap = test(data.test_pos_edge_index, data.test_neg_edge_index)
    print('Epoch: {:03d}, AUC: {:.4f}, AP: {:.4f}'.format(epoch, auc, ap))

    writer.add_scalar('auc train', auc, epoch)   # new line
    writer.add_scalar('ap train', ap, epoch)   # new line

Epoch: 001, AUC: 0.6307, AP: 0.6709
Epoch: 002, AUC: 0.6535, AP: 0.6918
Epoch: 003, AUC: 0.6667, AP: 0.7029
Epoch: 004, AUC: 0.6725, AP: 0.7078
Epoch: 005, AUC: 0.6759, AP: 0.7107
Epoch: 006, AUC: 0.6784, AP: 0.7131
Epoch: 007, AUC: 0.6803, AP: 0.7150
Epoch: 008, AUC: 0.6817, AP: 0.7167
Epoch: 009, AUC: 0.6828, AP: 0.7182
Epoch: 010, AUC: 0.6837, AP: 0.7198
Epoch: 011, AUC: 0.6846, AP: 0.7225
Epoch: 012, AUC: 0.6865, AP: 0.7253
Epoch: 013, AUC: 0.6869, AP: 0.7289
Epoch: 014, AUC: 0.6862, AP: 0.7317
Epoch: 015, AUC: 0.6848, AP: 0.7333
Epoch: 016, AUC: 0.6826, AP: 0.7348
Epoch: 017, AUC: 0.6811, AP: 0.7359
Epoch: 018, AUC: 0.6794, AP: 0.7367
Epoch: 019, AUC: 0.6774, AP: 0.7366
Epoch: 020, AUC: 0.6762, AP: 0.7367
Epoch: 021, AUC: 0.6751, AP: 0.7367
Epoch: 022, AUC: 0.6747, AP: 0.7370
Epoch: 023, AUC: 0.6740, AP: 0.7370
Epoch: 024, AUC: 0.6737, AP: 0.7371
Epoch: 025, AUC: 0.6736, AP: 0.7375
Epoch: 026, AUC: 0.6737, AP: 0.7378
Epoch: 027, AUC: 0.6742, AP: 0.7384
Epoch: 028, AUC: 0.6749, AP:

## Graph Variational AutoEncoder (GVAE)

In [23]:
from torch_geometric.nn import VGAE

In [ ]:
dataset = Planetoid('data', 'CiteSeer', transform=T.NormalizeFeatures())
data = dataset[0]
data.train_mask = data.val_mask = data.test_mask = data.y = None
data = train_test_split_edges(data)


class VariationalGCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(VariationalGCNEncoder, self).__init__()
        self.conv1 = GCNConv(
            in_channels, 2 * out_channels, cached=True
        )   # cached only for transductive learning
        self.conv_mu = GCNConv(2 * out_channels, out_channels, cached=True)
        self.conv_logstd = GCNConv(2 * out_channels, out_channels, cached=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

/home/kklepikov/_code_/PytorchGeometricTutorial/.venv/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


In [25]:
out_channels = 2
num_features = dataset.num_features
epochs = 300


model = VGAE(VariationalGCNEncoder(num_features, out_channels))  # new line

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
x = data.x.to(device)
train_pos_edge_index = data.train_pos_edge_index.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
def train():
    model.train()
    optimizer.zero_grad()
    z = model.encode(x, train_pos_edge_index)
    loss = model.recon_loss(z, train_pos_edge_index)

    loss = loss + (1 / data.num_nodes) * model.kl_loss()  # new line
    loss.backward()
    optimizer.step()
    return float(loss)


def test(pos_edge_index, neg_edge_index):
    model.eval()
    with torch.no_grad():
        z = model.encode(x, train_pos_edge_index)
    return model.test(z, pos_edge_index, neg_edge_index)

In [ ]:
writer = SummaryWriter('runs/VGAE_experiment_' + '2d_100_epochs')

for epoch in range(1, epochs + 1):
    loss = train()
    auc, ap = test(data.test_pos_edge_index, data.test_neg_edge_index)
    print('Epoch: {:03d}, AUC: {:.4f}, AP: {:.4f}'.format(epoch, auc, ap))

    writer.add_scalar('auc train', auc, epoch)   # new line
    writer.add_scalar('ap train', ap, epoch)   # new line

Epoch: 001, AUC: 0.6404, AP: 0.6697
Epoch: 002, AUC: 0.6491, AP: 0.6837
Epoch: 003, AUC: 0.6453, AP: 0.6809
Epoch: 004, AUC: 0.6426, AP: 0.6771
Epoch: 005, AUC: 0.6386, AP: 0.6738
Epoch: 006, AUC: 0.6327, AP: 0.6689
Epoch: 007, AUC: 0.6335, AP: 0.6685
Epoch: 008, AUC: 0.6386, AP: 0.6711
Epoch: 009, AUC: 0.6408, AP: 0.6724
Epoch: 010, AUC: 0.6425, AP: 0.6743
Epoch: 011, AUC: 0.6431, AP: 0.6745
Epoch: 012, AUC: 0.6433, AP: 0.6744
Epoch: 013, AUC: 0.6438, AP: 0.6746
Epoch: 014, AUC: 0.6441, AP: 0.6747
Epoch: 015, AUC: 0.6444, AP: 0.6749
Epoch: 016, AUC: 0.6449, AP: 0.6755
Epoch: 017, AUC: 0.6452, AP: 0.6757
Epoch: 018, AUC: 0.6459, AP: 0.6761
Epoch: 019, AUC: 0.6457, AP: 0.6759
Epoch: 020, AUC: 0.6450, AP: 0.6755
Epoch: 021, AUC: 0.6463, AP: 0.6771
Epoch: 022, AUC: 0.6478, AP: 0.6784
Epoch: 023, AUC: 0.6497, AP: 0.6813
Epoch: 024, AUC: 0.6515, AP: 0.6838
Epoch: 025, AUC: 0.6526, AP: 0.6855
Epoch: 026, AUC: 0.6536, AP: 0.6869
Epoch: 027, AUC: 0.6544, AP: 0.6877
Epoch: 028, AUC: 0.6553, AP: